In [1]:
import time
import numpy as np
import xesmf as xe
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

In [2]:
from dask.distributed import Client
c = Client(threads_per_worker=1)

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-215931' coro=<Client._gather.<locals>.wait() done, defined at /g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/client.py:2388> exception=AllExit()>
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/client.py", line 2397, in wait
    raise AllExit()
distributed.client.AllExit
ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-215932' coro=<Client._gather.<locals>.wait() done, defined at /g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/client.py:2388> exception=AllExit()>
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/client.py", line 2397, in wait
    raise AllExit()
distributed.client.AllExit
ERROR:

In [3]:
def f(t, amp, phase, err):
    return amp * np.cos(freq * t - phase)

In [4]:
# h2u, h2v
weights_base = Path("/g/data/nm03/lxy581/global_drag_coeff/")
weights_h2u = weights_base / "8km_h_to_8km_u_weights.nc"
weights_h2v = weights_base / "8km_h_to_8km_v_weights.nc"

In [5]:
chunks = 300
coords = ("xh", "xq", "yh", "yq")
coord_chunks = {v: chunks for v in coords}

In [6]:
output_base = Path("/scratch/nm03/lxy581/mom6/archive/tides_008_global_sigma_SAL_2layer_x02/output013")
output_interior = output_base / 'ocean_interior.nc'
output_static = output_base / "ocean_static.nc"
data = xr.open_dataset(output_interior, chunks=coord_chunks | {"time": 1})
static = xr.open_dataset(output_static, chunks=coord_chunks)

In [7]:
time_len = 236
freq = 2 * np.pi / 12.4206014

In [8]:
def interp_h2u(var, weights_h2u):
    ds_h = xr.Dataset(data_vars={'var_h2u': var,
                                },
                      coords={'time': var.time,
                              'lon': static.geolon,
                              'lat': static.geolat})
    ds_u = xr.Dataset({"lat": static.geolat_u, "lon": static.geolon_u})
    
    regridder_h2u = xe.Regridder(ds_h, ds_u, "bilinear", extrap_method="inverse_dist", reuse_weights=True, weights=
                                 weights_h2u)
    ds_u = regridder_h2u(ds_h)
    return ds_u['var_h2u']

In [9]:
def interp_h2v(var, weights_h2v):
    ds_h = xr.Dataset(data_vars={'var_h2v': var,
                                },
                      coords={'time': var.time,
                              'lon': static.geolon,
                              'lat': static.geolat})
    ds_v = xr.Dataset({"lat": static.geolat_v, "lon": static.geolon_v})
    
    regridder_h2v = xe.Regridder(ds_h, ds_v, "bilinear", extrap_method="inverse_dist", reuse_weights=True, weights=
                                 weights_h2v)
    ds_v = regridder_h2v(ds_h)
    return ds_v['var_h2v']

In [10]:
e_top = data.e.isel(zi=0,time=slice(None, time_len)).squeeze()
e_mid = data.e.isel(zi=1,time=slice(None, time_len)).squeeze()
e_bot = data.e.isel(zi=2,time=slice(None, time_len)).squeeze()

In [11]:
start_time = time.time()

In [12]:
e_top_u = interp_h2u(e_top, weights_h2u)
e_top_v = interp_h2v(e_top, weights_h2v)

In [13]:
end_time = time.time()
exe_time = float(end_time - start_time)
print("Execution time: %.1f seconds! \n" % exe_time)

Execution time: 106.1 seconds! 



In [14]:
e_mid_u = interp_h2u(e_mid, weights_h2u)
e_mid_v = interp_h2v(e_mid, weights_h2v)

In [15]:
end_time = time.time()
exe_time = float(end_time - start_time)
print("Execution time: %.1f seconds! \n" % exe_time)

Execution time: 197.3 seconds! 



In [16]:
e_bot_u = interp_h2u(e_bot, weights_h2u)
e_bot_v = interp_h2v(e_bot, weights_h2v)

In [17]:
end_time = time.time()
exe_time = float(end_time - start_time)
print("Execution time: %.1f seconds! \n" % exe_time)

Execution time: 293.9 seconds! 



#### check shape

In [18]:
print(e_top_u.shape, e_top_v.shape)
print(e_mid_u.shape, e_mid_v.shape)
print(e_bot_u.shape, e_bot_v.shape)

(236, 3270, 4321) (236, 3271, 4320)
(236, 3270, 4321) (236, 3271, 4320)
(236, 3270, 4321) (236, 3271, 4320)


#### barotropic velocities (thickness-weighted)

In [19]:
start_time = time.time()

In [20]:
uh_top = data.uo.isel(zl=0,time=slice(None, time_len)).squeeze() * (e_mid_u - e_top_u)
uh_bot = data.uo.isel(zl=1,time=slice(None, time_len)).squeeze() * (e_bot_u - e_mid_u)
vh_top = data.vo.isel(zl=0,time=slice(None, time_len)).squeeze() * (e_mid_v - e_top_v)
vh_bot = data.vo.isel(zl=1,time=slice(None, time_len)).squeeze() * (e_bot_v - e_mid_v)

In [21]:
end_time = time.time()
exe_time = float(end_time - start_time)
print("Execution time: %.1f seconds! \n" % exe_time)

Execution time: 0.4 seconds! 



In [22]:
bt_u = (uh_top + uh_bot) / (e_bot_u - e_top_u)
bt_v = (vh_top + vh_bot) / (e_bot_v - e_top_v)

In [23]:
print(bt_u.shape)
print(bt_v.shape)

(236, 3270, 4321)
(236, 3271, 4320)


In [24]:
# -1 means: use the full time dimension as one chunk.
bt_u = bt_u.chunk({"time": -1})
bt_u["time"] = np.arange(len(bt_u.time)) + 130*24
fit_x = bt_u.curvefit("time", f)

In [25]:
bt_v = bt_v.chunk({"time": -1})
bt_v["time"] = np.arange(len(bt_v.time)) + 130*24
fit_y = bt_v.curvefit("time", f)

In [26]:
amp_x = fit_x.curvefit_coefficients.isel(param=0)
pha_x = fit_x.curvefit_coefficients.isel(param=1)

amp_x_final = abs(amp_x)

pha_x_adjusted = xr.where(amp_x < 0, pha_x + np.pi, pha_x)
pha_x_adjusted = pha_x_adjusted % (2 * np.pi)
pha_x_final = xr.where(pha_x_adjusted > np.pi, pha_x_adjusted - 2*np.pi, pha_x_adjusted)

In [27]:
amp_y = fit_y.curvefit_coefficients.isel(param=0)
pha_y = fit_y.curvefit_coefficients.isel(param=1)

amp_y_final = abs(amp_y)

pha_y_adjusted = xr.where(amp_y < 0, pha_y + np.pi, pha_y)
pha_y_adjusted = pha_y_adjusted % (2 * np.pi)
pha_y_final = xr.where(pha_y_adjusted > np.pi, pha_y_adjusted - 2*np.pi, pha_y_adjusted)

In [28]:
start_time = time.time()

In [ ]:
est_data = xr.Dataset({"amp_x": amp_x_final,
                       "phase_x": pha_x_final,
                       "amp_y": amp_y_final,
                       "phase_y": pha_y_final,})

est_data.to_netcdf("/g/data/nm03/lxy581/evaluate/amp_phase/form_vel_amp_pha_test_013.nc")

2026-04-20 19:51:00,753 - distributed.worker - ERROR - 
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/asyncio/base_events.py", line 678, in run_until_complete
    self.run_forever()
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    self._run_once()
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/asyncio/base_events.py", line 1961, in _run_once
    event_list = self._selector.select(timeout)
                 ^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
end_time = time.time()
exe_time = float(end_time - start_time)
print("Execution time: %.1f seconds! \n" % exe_time)